## Correction of the reddening and extinction 

I use my pre-downloaded 3D Lallement map 

In [1]:
import os
import numpy as np 
import healpy as hp
import pandas as pd

In [ ]:
# Read the data
t = pd.read_parquet("./data/1kpc/source/1kpc_tot.parquet")

In [ ]:
NSIDE = 16   # resolution of Lallement map 

# Each star in the catalog is sorted into our sky grid (its pixel number is calculated)
# Then we calculate the number of unique pixels (should be NPIX or close to this)
hpix = hp.pixelfunc.ang2pix(NSIDE,np.pi/2+np.deg2rad(t['b']),np.deg2rad(t['l']),nest=True)
uhpix = np.unique(hpix)

In [ ]:
# Pre-calculate reddening grid 
# (download the reddening curve for each pixel from uhpix)

Egrid = []
delta_d = 5  # pc, distance step of Lallement map 

print('Getting extinction...')

for i in range(len(uhpix)):
    extinction_table = pd.read_csv(
        os.path.join('..','..','code_inner_usage','dereddening','Lallement','dustmap_data',
                     ''.join(('dust_hpix',str(int(uhpix[i])),'.csv'))
                     ), # change path if needed
        delimiter=';',names=['d','ebv','ed','eebv_min','eebv_max'])
    E_bv_c = [np.float64(k) for k in extinction_table['ebv'][1:]] # [1:] skips header
    Egrid.append(E_bv_c)
    
Egrid = np.array(Egrid,dtype=object)

In [ ]:
# For each star in the catalog, find 
# - index of its pixel in the array of uhpix and 
# - distance index corresponding to Lallement distance grid 

index_h = np.array([np.where(uhpix==i)[0][0] for i in hpix],dtype=np.int32)

# If distance and M_G columns are missing, add them
t['d'] = 1000/ t['parallax']    # inverse parallax distance, pc
t['M_G'] = t['phot_g_mean_mag'] - 5*np.log10(t['d']) + 5  # absolute magnitude
t['BP_RP'] = t['phot_bp_mean_mag'] - t['phot_rp_mean_mag']

index_d = np.array(t['d']//delta_d,dtype=np.int32)

In [ ]:
# Find reddening and extinction from the dust map 
#----------------------------------------------------------------

# If distance in the catalog is larger than the maxuimum distance available 
# for this line-of-sight in the dust map, we take reddening for the largest available distance
for i in range(len(index_d)):
    if index_d[i] >= len(Egrid[index_h[i]]) or index_d[i] < 0:
        index_d[i] = len(Egrid[index_h[i]]) - 1
        
# In all other cases we simply get reddening in the direction of (l,b) and at distance d. 
E_bv = np.array([Egrid[i][k] for i,k in zip(index_h,index_d)])  

print('Colour transformations...')
# Transformation coefficients can be found at http://stev.oapd.inaf.it/cgi-bin/cmd
# (after you submit a request for some isochrone, there will a be a small table 
# at the new page). Just use Gaia passbands.  
A_v = 3.1*E_bv
A_g = 0.83627*A_v
A_bp = 1.08337*A_v
A_rp = 0.63439*A_v

# Extinction must be positive all the time
# This was some old check, probably not needed
A_g[A_g < 0] = 0 
A_bp[A_bp < 0] = 0 
A_rp[A_rp < 0] = 0 

In [ ]:
# And save the table with the new columns
# Use the right input column - I apply dust map on the deblended G - GRP color 
t['G_RP0_lallement_sm'] = np.subtract(t['g_rp_deblended_sm'],A_g-A_rp)
t['BP_RP0_lallement'] = np.subtract(t['BP_RP'],A_bp-A_rp)


# And save
t.to_parquet("./data/1kpc/source/1kpc_tot_drd.parquet", index=False)
print('Done')